# Classify with a neural network

**What it does.** Train a convolutional network on cropped single-object images and score every object in the screen.

**When to use it.** When the phenotype is visible but hard to write down as a rule — the classifier learns it from labelled examples.

**What you get.** A trained model, a held-out performance report, and per-object scores merged back into the measurement database.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.deep_spacr.deep_spacr`

```
deep_spacr(settings=None)
```

Run the full spacr deep-learning pipeline: build dataset, train, apply model, merge predictions into the measurements DB.

In [ ]:
from spacr.deep_spacr import deep_spacr

## 3. Settings

`spacr.settings.get_train_test_model_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_train_test_model_settings

defaults = get_train_test_model_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (bool) - Use the AMSGrad variant of Adam/AdamW, which keeps a
    # running maximum of past squared gradients instead of their
    # decaying average so the effective step size never grows back.
    # Enable when training loss oscillates or stops converging with
    # plain Adam; it costs a little speed and memory. Only honoured by
    # optimizer_type 'adam' and 'adamw' - ignored by sgd, rmsprop,
    # nadam, radam and adagrad. Default True.
    'amsgrad': True,

    # (bool) - Expand the training split 8-fold by adding all four
    # 90-degree rotations of each crop plus their horizontal mirrors;
    # the validation and test splits are never augmented. Turn it on
    # when you have few annotated objects and validation accuracy lags
    # training accuracy. The expanded set is materialised in RAM, so
    # expect roughly 8x the memory and 8x the epoch time. Default False.
    'augment': True,

    # (int) - How many images are held and processed together in one
    # pass: field stacks during normalization and Cellpose segmentation,
    # crops per step during classifier training and activation maps.
    # Raising it speeds runs up but increases RAM/VRAM roughly linearly;
    # lower it on out-of-memory errors. Defaults: 50 for mask
    # generation, 64 for training.
    'batch_size': 64,

    # (str) - How skew between the training classes is corrected. 'none'
    # (the default) changes nothing but still prints the per-class
    # counts, the majority-over-minority ratio and a recommendation, so
    # the skew is never invisible. 'weighted_sampler' attaches a
    # WeightedRandomSampler with 1/n weights, drawing every class about
    # equally often; 'sqrt_weighted_sampler' uses 1/sqrt(n) for a
    # gentler pull that avoids showing a tiny class so often the model
    # memorises it; 'weighted_loss' leaves sampling alone and switches
    # loss_type to 'ce_weighted' instead. Resampling is applied to the
    # train loader only - validation and test keep the real prior so
    # their scores stay comparable to the screen.
    'class_balance': 'none',

    # (list) - Ordered class names. Each must exactly match a subfolder
    # under src/train and src/test; a name's position in this list
    # becomes its integer label, and the list length sets the width of
    # the classifier head. Training raises a FileNotFoundError listing
    # missing vs available folders if a name has no folder. Generate
    # Training Dataset overwrites this with the class names it actually
    # wrote to disk. Default ['nc','pc'].
    'classes': ['nc', 'pc'],

    # (bool) - Build the Classifier Evaluation workbench bundle from
    # out-of-fold predictions after Classify (CV): sample-level
    # predictions, confusion matrices, reliability curves, calibrated
    # probabilities, per-plate metrics, leakage reports and a manifest.
    # It requires cross_validation_folds >= 2; a single train/validation
    # split cannot produce unbiased out-of-fold diagnostics. Default
    # True. API: spacr.classifier_evaluation.evaluate_predictions.
    'classifier_evaluation': True,

    # (bool) - Enable k-fold validation for Classify. If
    # cross_validation_folds is 0 or 1, enabling this uses 5 folds. Use
    # cv_group_by='plate' to hold out whole plates, or 'well'/'field'
    # for within-plate validation without leaking related crops between
    # training and validation.
    'cross_validation_enabled': False,

    # (int) - Number of k-fold splits the vision classifier is trained
    # with in place of the single val_split hold-out. 0 (the default) or
    # 1 keeps today's one random split; 2 or more trains a fresh model
    # per fold, scores each on the fold it never saw, and reports the
    # mean together with the fold-to-fold standard deviation and range -
    # which is the only way to see whether one lucky split was
    # flattering the model. Costs roughly k times the training time.
    # Distinct from 'cross_validation', which is the regression
    # pipeline's own toggle.
    'cross_validation_folds': 0,

    # (str) - Filesystem path to a saved Cellpose model, loaded as
    # pretrained_model by the mask-finetune tool (analyze_plaques sets
    # it internally to the bundled plaque model). When set, model_type
    # is passed as None and diameter is passed as diam_mean (ignored by
    # Cellpose 4.x with a warning), but model_name is still read: it
    # selects the channel pair sent to model.eval - cyto2 -> [2,1],
    # nucleus -> [0,0], cyto -> [1,0], anything else [2,0], overridden
    # to [0,0] when grayscale is True. If the path does not exist the
    # run prints 'Custom model not found' and returns without segmenting
    # any image. None builds the stock model instead. Default None (the
    # classifier-training defaults set a same-named boolean that nothing
    # reads).
    'custom_model': False,

    # (str) - Path to a trained classifier artifact whose model weights
    # initialize a new fine-tuning run. The optimizer and epoch start
    # fresh. Leave empty to initialize from ImageNet or random weights
    # according to init_weights. Default ''.
    'custom_model_path': '',

    # (str) - Which metadata level is kept intact across generated
    # train/test data, the ordinary validation holdout and every CV
    # fold: 'well' (the default and the right choice for object crops),
    # 'field', 'plate', or 'none' for legacy per-object splitting. Crops
    # from one well share focus, illumination, seeding density and edge
    # effects, so letting them straddle a boundary lets the model
    # recognise the well instead of the phenotype and inflates every
    # score. The level is parsed from the crop filename, which spaCR
    # writes as plate_well_field_object.png.
    'cv_group_by': 'well',

    # (float) - Dropout probability (0-1) written into every existing
    # Dropout layer of the backbone and applied to a Dropout inserted
    # before the final linear classifier; 0 or None removes dropout
    # entirely. Raise it (0.2-0.5) when training accuracy runs well
    # ahead of validation accuracy; lower it when the model underfits
    # and training loss stalls high. Default 0.1.
    'dropout_rate': 0.1,

    # (int) - Stop training after this many consecutive epochs in which
    # validation accuracy fails to beat the best value so far; the best
    # checkpoint is still kept. 0 (default) disables it and always runs
    # the full 'epochs' budget. Set 10-20 on long runs to cut wasted
    # epochs once the model plateaus.
    'early_stopping_patience': 0,

    # (int) - Number of full passes over the training set. It also sets
    # the learning-rate schedule horizon - cosine anneals over exactly
    # this many epochs and step_lr drops every epochs/5 - so changing it
    # rescales the schedule. A checkpoint is always written on the final
    # epoch and every 100th. Raise it for small datasets and use
    # early_stopping_patience to cut runs short. Default 100.
    'epochs': 100,

    # (int) - Number of equal-width probability bins in reliability
    # curves and expected calibration error. Values around 10 balance
    # resolution against noise; use fewer bins for small validation sets
    # and more only when every class has many hundreds of out-of-fold
    # samples. Minimum 2, default 10. API:
    # spacr.classifier_evaluation.calibration_table.
    'evaluation_bins': 10,

    # (str) - Probability calibration written to the evaluation bundle.
    # 'temperature' cross-fits one scalar temperature per held-out fold
    # using all other out-of-fold predictions, so a sample never fits
    # its own calibrator; 'none' retains raw softmax probabilities.
    # Calibration changes reported probabilities, not the saved model
    # weights. Default 'temperature'. API:
    # spacr.classifier_evaluation.cross_calibrate_probabilities.
    'evaluation_calibration': 'temperature',

    # (bool) - Stop Classify (CV) before fitting a fold when the same
    # object, augmentation family, or protected cv_group_by identity
    # appears in both train and validation. False records the problem
    # and continues, which is useful only for diagnosing a legacy
    # dataset because its performance estimate remains invalid. Default
    # True. API: spacr.classifier_evaluation.audit_split_leakage.
    'evaluation_fail_on_leakage': True,

    # (float) - Class-balancing weight for focal loss (read only when
    # loss_type resolves to focal). In the single-logit binary path it
    # scales positives by alpha and negatives by 1-alpha, so raise it
    # toward 1 to emphasise a rare positive class; with two or more
    # output classes a plain float scales the whole loss uniformly.
    # Default None (no alpha weighting).
    'focal_alpha': None,

    # (float) - Focusing exponent in the focal-loss weight (1 -
    # p_t)^gamma, applied only when loss_type is focal. 0 reduces it to
    # plain cross-entropy; raising it (typically 1-5) down-weights crops
    # the model already classifies well and pushes gradient onto hard
    # ones. Default 2.0. Increase when one class dominates and training
    # stalls on easy examples.
    'focal_gamma': 2.0,

    # (bool) - Sum gradients over several batches before each optimizer
    # step instead of stepping on every batch, giving an effective batch
    # size of batch_size x gradient_accumulation_steps without extra GPU
    # memory. Enable when you had to shrink batch_size to fit in VRAM
    # and training is noisy. Leftover gradients are flushed at the end
    # of each epoch. Default True.
    'gradient_accumulation': True,

    # (int) - How many batches are summed per optimizer step when
    # gradient_accumulation is on; the loss is divided by this value so
    # gradient magnitude stays comparable. Effective batch size =
    # batch_size x this. Raise it (4-16) to emulate a larger batch on
    # limited VRAM, at the cost of fewer weight updates per epoch.
    # Ignored when gradient_accumulation is False. Default 4.
    'gradient_accumulation_steps': 4,

    # (int) - Side length in pixels of the centre crop taken from each
    # object PNG before it reaches the model. Images are cropped, not
    # rescaled, so a larger value zero-pads and a smaller one throws
    # away the object's edges. It is also the resolution the backbone is
    # built at, which matters for ViT/Swin/inception. Match it to the
    # crop size used when the dataset was generated. Default 224.
    'image_size': 224,

    # (bool) - Start the backbone from ImageNet-pretrained weights
    # instead of random initialisation; the spaCR classifier head bolted
    # on top is randomly initialised either way. Leave it on - transfer
    # learning converges in far fewer epochs on the small annotated sets
    # typical here. Turn it off only to train from scratch on a very
    # large dataset, or to measure how much pretraining contributes.
    # Default True.
    'init_weights': True,

    # (bool) - Intended to control whether extra checkpoints are written
    # mid-run when validation accuracy crosses 99, 98, 95 or 94 percent,
    # on top of the final-epoch save. It currently has no effect:
    # train_model passes that threshold list to the saver
    # unconditionally, so those checkpoints are written regardless of
    # this flag. Default True.
    'intermedeate_save': True,

    # (float) - Epsilon passed to cross-entropy when loss_type is
    # label_smoothing: each target keeps 1 - eps of its probability mass
    # and the rest is spread across the other classes. Raise it
    # (typically 0.05-0.2) when the model gets over-confident or
    # annotations are noisy; 0 disables. Ignored by every other loss
    # type. Default 0.1.
    'label_smoothing': 0.1,

    # (bool) - Audit the permanent train/ and test/ boundary before any
    # classifier fit. Checks plate/well/field/object lineage, exported
    # augmentation families and (when enabled) byte-identical renamed
    # copies. Default True. API:
    # spacr.classifier_evaluation.audit_dataset_splits.
    'leakage_audit_train_test': True,

    # (bool) - SHA-256 hash classifier images during leakage audits so
    # an identical crop copied or renamed across train/test or CV
    # boundaries is still detected. Reads files in 1 MiB chunks and
    # never decodes pixels. Default True. API:
    # spacr.classifier_evaluation.audit_cv_folds.
    'leakage_hash_content': True,

    # (bool) - Treat filenames that do not encode the protected
    # cv_group_by identity, and files that cannot be hashed, as a failed
    # audit rather than an advisory warning. Default True because
    # independence cannot be claimed when lineage is unknown. API:
    # spacr.classifier_evaluation.audit_split_leakage.
    'leakage_require_identity': True,

    # (float) - Step size passed to the optimizer. Too high and the loss
    # spikes or flatlines at chance; too low and training crawls or
    # settles in a poor minimum. 1e-3 suits training from scratch, while
    # 1e-4 to 1e-5 is safer when fine-tuning ImageNet weights
    # (init_weights=True). The chosen schedule decays this starting
    # value over the run. Default 0.001.
    'learning_rate': 0.0001,

    # (float) - Strength of the Menon-et-al. logit adjustment: tau *
    # log(class prior) is added to the logits during training, pulling
    # decisions toward rare classes. Only used when loss_type resolves
    # to logit_adjust_ce, which 'auto' picks when the smallest class is
    # under 10% of the data. Higher tau corrects harder; 0 disables.
    # Default 1.0.
    'logit_adjust_tau': 1.0,

    # (str) - Which loss build_loss constructs for the classifier. For a
    # 2+ class head the working values are 'focal_loss'/'focal_ce'
    # (down-weights easy examples), 'cross_entropy'/'ce',
    # 'label_smoothing'/'ce_smooth' (epsilon fixed at 0.1),
    # 'ce_weighted' (inverse-frequency class weights), 'logit_adjust_ce'
    # and 'asl'; 'binary_cross_entropy_with_logits'/'bce' is legal only
    # for a single-logit head and raises otherwise. 'auto' is rewritten
    # to 'cross_entropy' before training starts, so build_loss's
    # rare-class switch to logit_adjust_ce never fires from this path.
    # Reach for 'focal_loss' or 'ce_weighted' when one class dominates
    # and the model collapses to predicting it. Default 'focal_loss'
    # ('auto' only in deep_spacr_defaults).
    'loss_type': 'focal_loss',

    # (str) - Backbone architecture for the single-object image
    # classifier, passed to choose_model: any TorchVision classification
    # model name (resnet50, maxvit_t, densenet121, ...). An unrecognised
    # name is not fatal at call time - choose_model prints 'Invalid
    # model_type' and returns None, so training then fails; the special
    # name 'custom' passes the name check but raises
    # NotImplementedError. Bigger backbones capture subtler phenotypes
    # but cost VRAM and epochs, and the name becomes part of the output
    # model folder path (src/model/<model_type>/...). Default 'maxvit_t'
    # in the training pipelines; the activation-map tool defaults to
    # 'maxvit', and only that exact string triggers its automatic
    # target-layer pick; the Tk/Qt combo preselects 'resnet50'.
    'model_type': 'maxvit_t',

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where
    # -1 means every core. Raise it to shorten CPU-bound steps until RAM
    # or disk I/O saturates. Note the measure-and-crop pipeline
    # overrides your value with cpu_count()-4. Defaults vary by
    # pipeline: cpu_count()-4, -1, or None.
    'n_jobs': 30,

    # (int) - Number of inner grouped folds used inside every outer CV
    # fold. 0 (default) keeps the faster ordinary grouped CV; 2 or more
    # trains one inner model per fold, uses inner validation for early
    # stopping/model selection, ensembles those models, and evaluates
    # only once on the untouched outer fold. Runtime is approximately
    # outer_folds x inner_folds training runs, but the outer score is
    # not reused for tuning. API:
    # spacr.classifier_evaluation.nested_group_folds.
    'nested_cv_inner_folds': 0,

    # (bool) - Percentile-normalize each image channel (2nd to 98th
    # percentile, clipped to 0-1) before display or model input; in the
    # activation-map tool this rescales the image the CAM/saliency
    # heatmap is drawn over. Turn it on when raw channels are too dim to
    # read under the overlay. Affects display and input scaling only,
    # never stored pixels. Default True.
    'normalize': True,

    # (str) - PyTorch optimizer used by deep_spacr.train_model: 'adamw',
    # 'adam', 'adamax', 'sgd', 'rmsprop', 'nadam', 'radam', 'adagrad',
    # 'adadelta' or 'asgd'. AdamW is the robust fine-tuning default; SGD
    # can generalise better but usually needs more epochs. amsgrad
    # applies only to Adam/AdamW. API:
    # spacr.deep_spacr.train_model(optimizer_type=...). Default 'adamw'.
    'optimizer_type': 'adamw',

    # (bool) - Decode and hold the entire train/test image set in RAM up
    # front (loaded in parallel across all cores) and hand batches to
    # the GPU from page-locked memory. Enable when the dataset fits
    # comfortably in RAM and disk I/O is the bottleneck; disable for
    # large datasets or it will exhaust memory before the first epoch
    # even starts. Default False.
    'pin_memory': True,

    # (bool) - Render and save QC figures while the pipeline runs:
    # channel montages and Cellpose mask overlays during segmentation,
    # before/after filtration views and crop grids during measurement.
    # It adds figures per batch, so a full plate becomes much slower and
    # more memory-hungry; keep it for small or test_mode runs, which
    # force it on. Default False.
    'plot': True,

    # (str) - Path to a spaCR training artifact to continue exactly:
    # restores model, optimizer, scheduler, epoch, best score and
    # random-generator state. Use custom_model_path instead when only
    # the weights should be reused. Default ''.
    'resume_checkpoint': '',

    # (str) - Learning-rate scheduler used by
    # spacr.deep_spacr.train_model: 'cosine', 'cosine_warm_restarts',
    # 'reduce_lr_on_plateau', 'step_lr', 'exponential', 'linear', or
    # 'none'. Plateau reacts to validation loss; cosine and linear use
    # the epoch budget; warm restarts periodically raise the rate to
    # escape a narrow minimum. API: train_model(schedule=...). Default
    # 'cosine'.
    'schedule': 'cosine',

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (bool) - Write live PyTorch loss, accuracy, macro-F1 and
    # learning-rate events to dst/tensorboard while the vision model
    # trains. Open that folder with tensorboard --logdir PATH for an
    # interactive dashboard that can compare runs. The in-app zoomable
    # loss/accuracy monitor is controlled separately by plot. Default
    # True.
    'tensorboard': True,

    # (bool) - In classifier training, run the held-out evaluation pass
    # (combine with train, or use alone to score an existing model). In
    # the sequencing barcode mapper it means something different:
    # process only the first read chunk and print a preview, so you can
    # sanity-check the regex and barcode CSVs in seconds. Default False.
    'test': False,

    # (bool) - Whether to train the model.
    'train': True,

    # (list) - Which colour planes of each object crop the classifier
    # sees, chosen from 'r', 'g' and 'b'. Fewer channels means a smaller
    # input tensor and a model that cannot use the dropped stain, so
    # drop a channel only when it carries no signal for your phenotype.
    # The joined letters also become part of the saved model's filename.
    # Default ['r', 'g', 'b'].
    'train_channels': ['r', 'g', 'b'],

    # (bool) - Run the backbone's forward pass through
    # torch.utils.checkpoint: intermediate activations are discarded and
    # recomputed during the backward pass, trading extra compute for a
    # large drop in activation memory. Enable when a bigger batch_size
    # or image_size gives CUDA out-of-memory; disable for the fastest
    # epochs when VRAM is not the constraint. Default True.
    'use_checkpoint': True,

    # (float) - Fraction of src/train randomly held out as a validation
    # set each run (0.1 = 10 percent). The validation score drives
    # checkpoint selection, early stopping and the live training curves;
    # at 0 there is no validation loader, so checkpointing falls back to
    # training accuracy, which rewards memorisation. Raise it on small
    # datasets for a less noisy estimate. Default 0.1.
    'val_split': 0.1,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

    # (float) - L2 penalty applied to the weights on every optimizer
    # step (AdamW applies it decoupled from the gradient). Raise it,
    # toward 1e-3 to 1e-2, when validation loss climbs while training
    # loss keeps falling; lower it toward 0 when the model cannot fit
    # the training set at all. Every supported optimizer honours it.
    # Default 0.00001.
    'weight_decay': 1e-05,

}

# Fill in anything left unset, then check the source path.
settings = get_train_test_model_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
deep_spacr(settings)

## Where the output went

A trained model, a held-out performance report, and per-object scores merged back into the measurement database.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.